In [2]:
import accelforge as af
from scheduling.scheduler import *
from af_wrapper import *
import numpy as np


def run(
    einsums,
    compute_units,
    data_dependencies,
    latency_per_component_grid = None,
    total_latency_grid = None,
    actions_grid = None,
    memory_name = None,
    shared_memory_info = None,
):
    schedule, min_latency = best_schedule(
        einsums,
        compute_units,
        shared_memory_info,
        data_dependencies,
        latency_per_component_grid,
        total_latency_grid,
        actions_grid,
        memory_name
    )
    return schedule, min_latency


In [4]:
data_dependencies = {
    "Matmul1": [],
    "Matmul2": []
}
compute_units = ['core1-GlobalBuffer-50', 'core2-GlobalBuffer-50']
einsums = data_dependencies.keys()
memory_name = "MainMemory"

In [5]:
%%capture grid_out
# this is for bw-aware version
# RUN TO GENERATE ACCELFORGE VALUES.
# To skip recomputation when running with fresh kernel, comment out
# this cell and use the below assignments instead.

workload = "workload/mm/mm.yaml"

(grid_lats, grid_mems, grid_maps) = af_memoizable_grid(
    einsums, 
    compute_units,
    lambda einsum: "workload/mm/"+einsum+".yaml",
    lambda sub_arch: "arch/eyeriss-dual-core-identical/"+sub_arch+".yaml"
)

In [6]:
grid_lats

{('core1-GlobalBuffer-50', 'Matmul1'): {'MAC1': np.float32(2.6843545),
  'InputScratchpad1': np.float32(3.0167785),
  'GlobalBuffer': np.float64(0.77627392),
  'MainMemory': np.float32(2.0237517),
  'WeightScratchpad1': np.float32(0.8769806),
  'OutputScratchpad1': np.float32(3.199435)},
 ('core1-GlobalBuffer-50', 'Matmul2'): {'MAC1': np.float32(2.6843545),
  'InputScratchpad1': np.float32(3.0167785),
  'GlobalBuffer': np.float32(0.79675394),
  'MainMemory': np.float32(2.0132658),
  'WeightScratchpad1': np.float32(0.8769806),
  'OutputScratchpad1': np.float32(3.199435)},
 ('core2-GlobalBuffer-50', 'Matmul1'): {'MAC2': np.float32(2.6843545),
  'InputScratchpad2': np.float32(3.0167785),
  'GlobalBuffer': np.float32(0.79691774),
  'MainMemory': np.float32(2.0237517),
  'WeightScratchpad2': np.float32(0.8769806),
  'OutputScratchpad2': np.float32(3.199435)},
 ('core2-GlobalBuffer-50', 'Matmul2'): {'MAC2': np.float32(2.6843545),
  'InputScratchpad2': np.float32(3.0167785),
  'GlobalBuffer':

In [7]:
grid_mems

{('core1-GlobalBuffer-50',
  'Matmul1'): {('InputScratchpad1',
   'read'): np.float64(274877906944.0), ('InputScratchpad1', 'write'): np.float32(2.748779e+11), ('GlobalBuffer',
   'read'): np.float32(4.831838e+09), ('GlobalBuffer',
   'write'): np.float32(1.3631488e+08), ('MainMemory',
   'read'): np.float32(1.0804527e+10), ('MainMemory',
   'write'): np.float64(2147483648.0), ('WeightScratchpad1',
   'read'): np.float64(274877906944.0), ('WeightScratchpad1',
   'write'): np.float32(2.748779e+11), ('OutputScratchpad1',
   'read'): np.float32(2.915209e+11), ('OutputScratchpad1',
   'write'): np.float32(2.915209e+11), ('MAC1',
   'compute'): np.float64(34359738368.0)},
 ('core1-GlobalBuffer-50',
  'Matmul2'): {('InputScratchpad1',
   'read'): np.float64(274877906944.0), ('InputScratchpad1',
   'write'): np.float32(2.748779e+11), ('GlobalBuffer',
   'read'): np.float32(4.8978985e+09), ('GlobalBuffer',
   'write'): np.float32(2.013266e+08), ('MainMemory',
   'read'): np.float32(1.0737418e+

In [10]:
filtered = {
    outer_key: {
        inner_key: value
        for inner_key, value in inner_dict.items()
        if 'MainMemory' in inner_key
    }
    for outer_key, inner_dict in grid_mems.items()
}

filtered

{('core1-GlobalBuffer-50',
  'Matmul1'): {('MainMemory',
   'read'): np.float32(1.0804527e+10), ('MainMemory', 'write'): np.float64(2147483648.0)},
 ('core1-GlobalBuffer-50',
  'Matmul2'): {('MainMemory',
   'read'): np.float32(1.0737418e+10), ('MainMemory', 'write'): np.float64(2147483648.0)},
 ('core2-GlobalBuffer-50',
  'Matmul1'): {('MainMemory',
   'read'): np.float32(1.0804527e+10), ('MainMemory', 'write'): np.float64(2147483648.0)},
 ('core2-GlobalBuffer-50',
  'Matmul2'): {('MainMemory', 'read'): np.float32(1.832072e+10), ('MainMemory',
   'write'): np.float64(1073741824.0)}}

In [12]:
grid_maps[('core1-GlobalBuffer-50','Matmul1')].actions(per_component=True, per_einsum=True)

{('Matmul1', 'InputScratchpad1', 'read'): np.float64(274877906944.0),
 ('Matmul1', 'InputScratchpad1', 'write'): np.float32(2.748779e+11),
 ('Matmul1', 'GlobalBuffer', 'read'): np.float32(4.831838e+09),
 ('Matmul1', 'GlobalBuffer', 'write'): np.float32(1.3631488e+08),
 ('Matmul1', 'MainMemory', 'read'): np.float32(1.0804527e+10),
 ('Matmul1', 'MainMemory', 'write'): np.float64(2147483648.0),
 ('Matmul1', 'WeightScratchpad1', 'read'): np.float64(274877906944.0),
 ('Matmul1', 'WeightScratchpad1', 'write'): np.float32(2.748779e+11),
 ('Matmul1', 'OutputScratchpad1', 'read'): np.float32(2.915209e+11),
 ('Matmul1', 'OutputScratchpad1', 'write'): np.float32(2.915209e+11),
 ('Matmul1', 'MAC1', 'compute'): np.float64(34359738368.0)}

In [13]:
grid_maps[('core2-GlobalBuffer-50','Matmul2')].actions(per_component=True, per_einsum=True)

{('Matmul2', 'InputScratchpad2', 'read'): np.float64(274877906944.0),
 ('Matmul2', 'InputScratchpad2', 'write'): np.float32(2.748779e+11),
 ('Matmul2', 'GlobalBuffer', 'read'): np.float32(4.8811213e+09),
 ('Matmul2', 'GlobalBuffer', 'write'): np.float32(3.1981568e+08),
 ('Matmul2', 'MainMemory', 'read'): np.float32(1.832072e+10),
 ('Matmul2', 'MainMemory', 'write'): np.float64(1073741824.0),
 ('Matmul2', 'WeightScratchpad2', 'read'): np.float64(274877906944.0),
 ('Matmul2', 'WeightScratchpad2', 'write'): np.float32(2.748779e+11),
 ('Matmul2', 'OutputScratchpad2', 'read'): np.float32(2.915209e+11),
 ('Matmul2', 'OutputScratchpad2', 'write'): np.float32(2.915209e+11),
 ('Matmul2', 'MAC2', 'compute'): np.float64(34359738368.0)}